# Milestone 4: PCA Model + Unsupervised Acoustic-State Analysis

This notebook is designed for the DSC 232R Beehive Sound Health Monitoring project. It adds a second model using **dimensionality reduction** and a unique unsupervised analysis that treats beehive sound clips as recurring **acoustic states**.

Main outputs:

1. PCA explained-variance plot  
2. PCA + Logistic Regression supervised model  
3. Training vs. test fitting graph across PCA dimensions  
4. Prediction analysis with correct classifications, false positives, and false negatives  
5. PCA + KMeans unsupervised acoustic-state analysis  
6. Silhouette-score plot for acoustic-state selection  
7. Cluster interpretation tables for the README  
8. Speedup-analysis table template

**Before running:** edit the configuration cell below so it points to your existing preprocessed Spark dataset and uses the correct target column name.

## 1. Spark Setup

Run this notebook on SDSC Expanse. Do not run the final submitted version locally. The Spark UI screenshot should show the executors/workers active during preprocessing or model training.

If your allocation is different, update the executor memory and executor count using the course formula:

- Executor instances = total cores - 1
- Executor memory = (total memory - driver memory) / executor instances

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Beehive_Milestone4_PCA_Acoustic_States")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.instances", "7")
    .config("spark.executor.memory", "18g")
    .config("spark.sql.shuffle.partitions", "56")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    VectorAssembler,
    StandardScaler,
    StringIndexer,
    IndexToString,
    PCA
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.linalg import VectorUDT

import os
import time
import math
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)

## 2. Configuration

Update these values to match your repo paths and final preprocessed data.

Expected input options:

- Best option: a Spark Parquet file from Milestone 3 containing one row per audio clip.
- Acceptable option: a CSV/Parquet with numeric audio features and a target label.
- If a `features` vector column already exists, the notebook will use it.
- If no `features` column exists, the notebook will assemble numeric feature columns automatically after excluding labels and identifiers.

In [ ]:
# -------------------------------------------------------------------
# EDIT THESE VALUES BEFORE RUNNING
# -------------------------------------------------------------------

# Path to your Milestone 3 preprocessed Spark dataset.
# Recommended: save your Part 3 final dataframe as Parquet and point here.
DATA_PATH = "../data/processed/preprocessed_features.parquet"
DATA_FORMAT = "parquet"  # "parquet" or "csv"

# Main target column for supervised learning.
# Common possibilities in this project might be: "health", "label", "target", "queen_presence".
TARGET_COL = "target"

# Optional ID columns used only for displaying examples; they are not used as features.
PREFERRED_ID_COLS = ["file name", "file_name", "filename", "path", "source_path"]

# Optional audio feature columns for the acoustic-state cluster profile.
# The notebook will use the subset that actually exists in your dataframe.
AUDIO_FEATURE_COLS = [
    "rms", "rms_energy", "spectral_centroid", "spectral_rolloff", "zero_crossing_rate",
    "mfcc_1", "mfcc_2", "mfcc_3", "mfcc_4", "mfcc_5", "mfcc_6", "mfcc_7",
    "mfcc_8", "mfcc_9", "mfcc_10", "mfcc_11", "mfcc_12", "mfcc_13"
]

# PCA dimensions to compare for the supervised model.
PCA_K_VALUES = [2, 5, 10, 15, 20, 30]

# KMeans cluster counts to compare for the unsupervised acoustic-state analysis.
KMEANS_K_VALUES = [2, 3, 4, 5, 6, 8, 10]

# Output directory for plots and small result files.
OUTPUT_DIR = "../figures/milestone4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("TARGET_COL:", TARGET_COL)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 3. Load Data

This section loads your existing preprocessed data. Keep the schema output in the committed notebook so the grader can see what columns were available.

In [ ]:
if DATA_FORMAT.lower() == "parquet":
    raw_df = spark.read.parquet(DATA_PATH)
elif DATA_FORMAT.lower() == "csv":
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(DATA_PATH)
    )
else:
    raise ValueError("DATA_FORMAT must be either 'parquet' or 'csv'.")

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))
raw_df.printSchema()
raw_df.show(5, truncate=False)

In [ ]:
if TARGET_COL not in raw_df.columns:
    raise ValueError(
        f"TARGET_COL='{TARGET_COL}' was not found. Available columns: {raw_df.columns}"
    )

id_cols = [c for c in PREFERRED_ID_COLS if c in raw_df.columns]
print("Identifier columns available for examples:", id_cols)

print("Target distribution:")
raw_df.groupBy(TARGET_COL).count().orderBy(F.desc("count")).show(50, truncate=False)

## 4. Build a Modeling DataFrame

This cell standardizes the input for the rest of the notebook:

- `label`: numeric class label for supervised learning
- `features`: Spark ML vector of model features
- `weight`: class weight to reduce class-imbalance effects

If your Milestone 3 pipeline already created `features`, this notebook reuses that vector. Otherwise, it imputes numeric columns, assembles them into a vector, and scales them.

In [ ]:
# Detect whether the target is already numeric.
target_dtype = dict(raw_df.dtypes)[TARGET_COL]
numeric_spark_types = {"tinyint", "smallint", "int", "bigint", "float", "double", "decimal"}
target_is_numeric = any(target_dtype.startswith(t) for t in numeric_spark_types)

working_df = raw_df

# Label handling
if target_is_numeric:
    working_df = working_df.withColumn("label", F.col(TARGET_COL).cast("double"))
    label_indexer_model = None
    label_names = None
else:
    label_indexer = StringIndexer(
        inputCol=TARGET_COL,
        outputCol="label",
        handleInvalid="keep"
    )
    label_indexer_model = label_indexer.fit(working_df)
    working_df = label_indexer_model.transform(working_df)
    label_names = list(label_indexer_model.labels)
    print("StringIndexer label mapping:")
    for i, name in enumerate(label_names):
        print(f"  {i}.0 -> {name}")

# Drop rows without a label.
working_df = working_df.filter(F.col("label").isNotNull())

# Use existing features vector if present.
features_schema = working_df.schema["features"].dataType if "features" in working_df.columns else None
has_vector_features = isinstance(features_schema, VectorUDT)

if has_vector_features:
    model_df = working_df
    print("Using existing Spark ML vector column: features")
else:
    excluded_cols = set([TARGET_COL, "label", "prediction", "probability", "rawPrediction", "features", "scaled_features"] + id_cols)
    numeric_cols = [
        c for c, dtype in working_df.dtypes
        if c not in excluded_cols and any(dtype.startswith(t) for t in numeric_spark_types)
    ]
    if len(numeric_cols) == 0:
        raise ValueError("No numeric feature columns found and no existing 'features' vector column exists.")

    print("Assembling numeric feature columns:")
    for c in numeric_cols:
        print(" -", c)

    imputed_cols = [f"{c}_imputed" for c in numeric_cols]

    imputer = Imputer(
        inputCols=numeric_cols,
        outputCols=imputed_cols,
        strategy="median"
    )

    assembler = VectorAssembler(
        inputCols=imputed_cols,
        outputCol="raw_features",
        handleInvalid="keep"
    )

    scaler = StandardScaler(
        inputCol="raw_features",
        outputCol="features",
        withMean=True,
        withStd=True
    )

    feature_pipeline = Pipeline(stages=[imputer, assembler, scaler])
    feature_model = feature_pipeline.fit(working_df)
    model_df = feature_model.transform(working_df)

# Keep useful columns.
keep_cols = list(dict.fromkeys(id_cols + [TARGET_COL, "label", "features"] + [c for c in AUDIO_FEATURE_COLS if c in model_df.columns]))
model_df = model_df.select(*keep_cols).dropna(subset=["label", "features"])

print("Modeling dataframe rows:", model_df.count())
model_df.select(id_cols + [TARGET_COL, "label"] if id_cols else [TARGET_COL, "label"]).show(10, truncate=False)

In [ ]:
# Create class weights from the full modeling dataframe.
label_counts = model_df.groupBy("label").count()
total_count = model_df.count()
num_classes = label_counts.count()

weights_df = label_counts.withColumn(
    "weight",
    F.lit(total_count) / (F.lit(num_classes) * F.col("count"))
).select("label", "weight")

print("Class weights:")
weights_df.orderBy("label").show(truncate=False)

model_df = model_df.join(weights_df, on="label", how="left")
model_df.cache()
print("Cached model_df rows:", model_df.count())

## 5. Train/Test Split

The same split is used for all PCA + Logistic Regression comparisons so the fitting graph is fair.

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

print("Train label distribution:")
train_df.groupBy("label").count().orderBy("label").show()

print("Test label distribution:")
test_df.groupBy("label").count().orderBy("label").show()

## 6. PCA + Logistic Regression: Fitting Analysis

This is the main Milestone 4 supervised model. It uses PCA for dimensionality reduction, then trains Logistic Regression on the reduced feature vectors.

We compare several values of `k`, the number of PCA components. This gives a fitting graph:

- Too few components: likely underfitting because the model discards too much information.
- Moderate components: best expected test performance.
- Too many components: possible overfitting if training error improves while test error worsens.

In [ ]:
# Determine feature vector size so PCA k values do not exceed the input dimension.
first_features = model_df.select("features").first()["features"]
feature_dim = len(first_features)
valid_pca_ks = sorted(set([k for k in PCA_K_VALUES if 1 <= k <= feature_dim]))

# Always include the full feature dimension if the configured k values were all too large.
if not valid_pca_ks:
    valid_pca_ks = [min(feature_dim, 2)]

print("Feature dimension:", feature_dim)
print("PCA k values to evaluate:", valid_pca_ks)

In [ ]:
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

weighted_precision_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

weighted_recall_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

pca_lr_results = []
pca_lr_models = {}

for k in valid_pca_ks:
    print(f"Training PCA + Logistic Regression with k={k}...")
    start_time = time.perf_counter()

    pca = PCA(
        k=k,
        inputCol="features",
        outputCol="pca_features"
    )

    lr = LogisticRegression(
        featuresCol="pca_features",
        labelCol="label",
        weightCol="weight",
        maxIter=100,
        regParam=0.01,
        elasticNetParam=0.0,
        family="auto"
    )

    pipeline = Pipeline(stages=[pca, lr])
    model = pipeline.fit(train_df)

    train_pred = model.transform(train_df)
    test_pred = model.transform(test_df)

    train_accuracy = acc_eval.evaluate(train_pred)
    test_accuracy = acc_eval.evaluate(test_pred)
    train_f1 = f1_eval.evaluate(train_pred)
    test_f1 = f1_eval.evaluate(test_pred)
    test_precision = weighted_precision_eval.evaluate(test_pred)
    test_recall = weighted_recall_eval.evaluate(test_pred)

    pca_model = model.stages[0]
    explained_variance = float(sum(pca_model.explainedVariance))

    elapsed = time.perf_counter() - start_time

    pca_lr_results.append({
        "pca_k": int(k),
        "explained_variance": explained_variance,
        "train_accuracy": float(train_accuracy),
        "test_accuracy": float(test_accuracy),
        "train_f1": float(train_f1),
        "test_f1": float(test_f1),
        "test_weighted_precision": float(test_precision),
        "test_weighted_recall": float(test_recall),
        "train_error": float(1.0 - train_accuracy),
        "test_error": float(1.0 - test_accuracy),
        "accuracy_gap": float(train_accuracy - test_accuracy),
        "wall_clock_sec": float(elapsed)
    })

    pca_lr_models[k] = model

pca_lr_results_df = spark.createDataFrame(pca_lr_results)
pca_lr_results_df.orderBy("pca_k").show(truncate=False)

In [ ]:
# Save model comparison table for README use.
comparison_path = os.path.join(OUTPUT_DIR, "pca_lr_model_comparison")
(
    pca_lr_results_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(comparison_path)
)
print("Saved model comparison table to:", comparison_path)

### PCA Explained Variance Plot

This plot shows how much of the original feature variation is retained as the number of principal components increases.

In [ ]:
rows = pca_lr_results_df.orderBy("pca_k").collect()
ks = [r["pca_k"] for r in rows]
explained = [r["explained_variance"] for r in rows]

plt.figure(figsize=(8, 5))
plt.plot(ks, explained, marker="o")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance")
plt.grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "pca_explained_variance.png")
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)

### Fitting Graph: Training vs. Test Error

This graph directly addresses underfitting vs. overfitting. Include it in the final README.

In [ ]:
rows = pca_lr_results_df.orderBy("pca_k").collect()
ks = [r["pca_k"] for r in rows]
train_errors = [r["train_error"] for r in rows]
test_errors = [r["test_error"] for r in rows]

plt.figure(figsize=(8, 5))
plt.plot(ks, train_errors, marker="o", label="Training error")
plt.plot(ks, test_errors, marker="s", label="Test error")
plt.xlabel("Number of PCA components")
plt.ylabel("Error rate")
plt.title("PCA + Logistic Regression Fitting Analysis")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "pca_lr_fitting_curve.png")
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)

In [ ]:
# Select the best model by test F1. If tied, prefer lower test error and fewer PCA components.
best_row = (
    pca_lr_results_df
    .orderBy(F.desc("test_f1"), F.asc("test_error"), F.asc("pca_k"))
    .first()
)

best_k = best_row["pca_k"]
best_model = pca_lr_models[best_k]

print("Best PCA k:", best_k)
print("Best model metrics:")
for key in best_row.asDict():
    print(f"  {key}: {best_row[key]}")

## 7. Prediction Analysis: Correct, False Positive, False Negative

The final README should include examples of predictions from the test set. This section creates:

- a confusion matrix
- examples of correct predictions
- examples of false positives
- examples of false negatives

For binary classification, choose the positive class carefully. If your label mapping is string-based, look at the mapping printed earlier.

In [ ]:
# Transform test set with the best model.
test_pred = best_model.transform(test_df).cache()

# Add human-readable predicted labels if StringIndexer was used.
if label_names is not None:
    converter = IndexToString(
        inputCol="prediction",
        outputCol="prediction_label",
        labels=label_names
    )
    test_pred = converter.transform(test_pred)
else:
    test_pred = test_pred.withColumn("prediction_label", F.col("prediction").cast("string"))

# Also make target_label human-readable.
if label_names is not None:
    mapping_expr = F.create_map(*[x for i, name in enumerate(label_names) for x in (F.lit(float(i)), F.lit(name))])
    test_pred = test_pred.withColumn("target_label", mapping_expr[F.col("label")])
else:
    test_pred = test_pred.withColumn("target_label", F.col("label").cast("string"))

print("Best model test metrics:")
print("Accuracy:", acc_eval.evaluate(test_pred))
print("Weighted F1:", f1_eval.evaluate(test_pred))
print("Weighted precision:", weighted_precision_eval.evaluate(test_pred))
print("Weighted recall:", weighted_recall_eval.evaluate(test_pred))

In [ ]:
confusion_df = (
    test_pred
    .groupBy("target_label", "prediction_label")
    .count()
    .orderBy("target_label", "prediction_label")
)

confusion_df.show(100, truncate=False)

confusion_path = os.path.join(OUTPUT_DIR, "pca_lr_confusion_matrix")
(
    confusion_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(confusion_path)
)
print("Saved confusion matrix table to:", confusion_path)

In [ ]:
# Plot confusion matrix without collecting the full prediction dataset.
conf_rows = confusion_df.collect()
target_labels = sorted({r["target_label"] for r in conf_rows})
pred_labels = sorted({r["prediction_label"] for r in conf_rows})

matrix = [[0 for _ in pred_labels] for _ in target_labels]
target_index = {name: i for i, name in enumerate(target_labels)}
pred_index = {name: i for i, name in enumerate(pred_labels)}

for r in conf_rows:
    matrix[target_index[r["target_label"]]][pred_index[r["prediction_label"]]] = r["count"]

plt.figure(figsize=(8, 6))
plt.imshow(matrix)
plt.xticks(range(len(pred_labels)), pred_labels, rotation=45, ha="right")
plt.yticks(range(len(target_labels)), target_labels)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("PCA + Logistic Regression Confusion Matrix")

for i in range(len(target_labels)):
    for j in range(len(pred_labels)):
        plt.text(j, i, str(matrix[i][j]), ha="center", va="center")

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "pca_lr_confusion_matrix.png")
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)

In [ ]:
# Choose a focus positive class for FP/FN analysis.
# For binary classification, update this if needed after reading the label mapping.
# If unsure, choose the minority class, because it is usually the more important class to analyze.
label_counts_test = test_pred.groupBy("label", "target_label").count().orderBy("count").collect()
FOCUS_LABEL = float(label_counts_test[0]["label"])
FOCUS_LABEL_NAME = label_counts_test[0]["target_label"]

print("Focus positive class for FP/FN analysis:", FOCUS_LABEL, FOCUS_LABEL_NAME)

prediction_cases = (
    test_pred
    .withColumn(
        "case_type",
        F.when(F.col("label") == F.col("prediction"), F.lit("correct"))
         .when((F.col("label") != FOCUS_LABEL) & (F.col("prediction") == FOCUS_LABEL), F.lit("false_positive"))
         .when((F.col("label") == FOCUS_LABEL) & (F.col("prediction") != FOCUS_LABEL), F.lit("false_negative"))
         .otherwise(F.lit("other_error"))
    )
)

prediction_cases.groupBy("case_type").count().orderBy("case_type").show(truncate=False)

In [ ]:
example_cols = id_cols + [TARGET_COL, "target_label", "prediction_label", "case_type", "probability"]
example_cols = [c for c in example_cols if c in prediction_cases.columns]

print("Correct prediction examples:")
prediction_cases.filter(F.col("case_type") == "correct").select(example_cols).show(5, truncate=False)

print("False positive examples:")
prediction_cases.filter(F.col("case_type") == "false_positive").select(example_cols).show(5, truncate=False)

print("False negative examples:")
prediction_cases.filter(F.col("case_type") == "false_negative").select(example_cols).show(5, truncate=False)

## 8. Unsupervised Acoustic-State Analysis: Learning the “Language” of Hives

This is the unique analysis for the final project.

The supervised model asks: **Can known hive labels be predicted?**  
The unsupervised model asks: **Do beehive recordings naturally organize into recurring sound states?**

We use PCA-reduced features followed by KMeans clustering. Then we compare clusters against known labels to interpret whether some acoustic states correspond to specific hive conditions.

In [ ]:
# Use 5 PCA components for acoustic-state discovery, unless the feature dimension is smaller.
cluster_pca_k = min(5, feature_dim)

pca_for_clusters = PCA(
    k=cluster_pca_k,
    inputCol="features",
    outputCol="pca_cluster_features"
)

pca_cluster_model = pca_for_clusters.fit(model_df)
cluster_input = pca_cluster_model.transform(model_df).cache()

print("Cluster PCA k:", cluster_pca_k)
print("Cluster PCA explained variance:", float(sum(pca_cluster_model.explainedVariance)))
cluster_input.select(id_cols + [TARGET_COL, "label", "pca_cluster_features"] if id_cols else [TARGET_COL, "label", "pca_cluster_features"]).show(5, truncate=False)

In [ ]:
cluster_results = []
cluster_models = {}

for k in KMEANS_K_VALUES:
    if k < 2:
        continue
    print(f"Training KMeans with k={k} acoustic states...")
    start_time = time.perf_counter()

    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="pca_cluster_features",
        predictionCol="acoustic_state"
    )

    km_model = kmeans.fit(cluster_input)
    clustered_tmp = km_model.transform(cluster_input)

    evaluator = ClusteringEvaluator(
        featuresCol="pca_cluster_features",
        predictionCol="acoustic_state",
        metricName="silhouette",
        distanceMeasure="squaredEuclidean"
    )

    silhouette = evaluator.evaluate(clustered_tmp)
    elapsed = time.perf_counter() - start_time

    cluster_results.append({
        "k": int(k),
        "silhouette": float(silhouette),
        "wall_clock_sec": float(elapsed)
    })
    cluster_models[k] = km_model

cluster_results_df = spark.createDataFrame(cluster_results)
cluster_results_df.orderBy("k").show(truncate=False)

In [ ]:
rows = cluster_results_df.orderBy("k").collect()
k_vals = [r["k"] for r in rows]
sil_vals = [r["silhouette"] for r in rows]

plt.figure(figsize=(8, 5))
plt.plot(k_vals, sil_vals, marker="o")
plt.xlabel("Number of acoustic states")
plt.ylabel("Silhouette score")
plt.title("KMeans Acoustic-State Selection")
plt.grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "acoustic_state_silhouette.png")
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)

In [ ]:
# Select best KMeans by silhouette. You can override this if a nearby k is more interpretable.
best_cluster_row = cluster_results_df.orderBy(F.desc("silhouette"), F.asc("k")).first()
best_cluster_k = best_cluster_row["k"]
best_cluster_model = cluster_models[best_cluster_k]

clustered = best_cluster_model.transform(cluster_input).cache()

print("Best number of acoustic states:", best_cluster_k)
print("Best silhouette score:", best_cluster_row["silhouette"])

In [ ]:
print("Acoustic state sizes:")
cluster_sizes = clustered.groupBy("acoustic_state").count().orderBy("acoustic_state")
cluster_sizes.show(truncate=False)

print("Target-label mix inside each acoustic state:")
cluster_label_mix = (
    clustered
    .groupBy("acoustic_state", TARGET_COL)
    .count()
    .orderBy("acoustic_state", F.desc("count"))
)
cluster_label_mix.show(100, truncate=False)

cluster_mix_path = os.path.join(OUTPUT_DIR, "acoustic_state_label_mix")
(
    cluster_label_mix
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(cluster_mix_path)
)
print("Saved acoustic state label mix table to:", cluster_mix_path)

In [ ]:
# Optional: profile acoustic states by raw audio features if those columns are available.
audio_cols_available = [c for c in AUDIO_FEATURE_COLS if c in clustered.columns]
print("Audio feature columns available for cluster profiling:", audio_cols_available)

if audio_cols_available:
    cluster_profiles = clustered.groupBy("acoustic_state").agg(
        F.count("*").alias("n"),
        *[F.avg(c).alias(f"avg_{c}") for c in audio_cols_available]
    ).orderBy("acoustic_state")

    cluster_profiles.show(truncate=False)

    profile_path = os.path.join(OUTPUT_DIR, "acoustic_state_profiles")
    (
        cluster_profiles
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(profile_path)
    )
    print("Saved acoustic state profiles to:", profile_path)
else:
    print("No raw audio feature columns found. Cluster interpretation will use PCA components and label mix only.")

### Acoustic-State Interpretation Notes for README

After running the cells above, write a short interpretation using the actual cluster tables. A strong structure is:

1. Report the best silhouette score and selected number of acoustic states.
2. Describe the largest clusters.
3. Identify whether any acoustic state is dominated by one health/target label.
4. Explain whether this supports the idea that hive conditions have distinctive sound signatures.
5. Acknowledge limits: clusters are not labels; they are unsupervised groupings and need biological validation.

## 9. Speedup Analysis Template

The course asks for a 1-executor baseline and a full-executor run. Because changing executor settings requires restarting Spark, do this as two separate notebook runs:

1. Run with `spark.executor.instances = 1`; record the model-training time.
2. Restart the kernel.
3. Run with the full executor configuration; record the same model-training time.

Use the same data, split, and model settings for both runs.

In [ ]:
# Fill these in after running the same representative operation under both configurations.
# Use the PCA + Logistic Regression loop or the best final model training as the representative operation.

speedup_measurements = [
    {"executors": 1, "time_sec": None},   # replace None with measured baseline time
    {"executors": 7, "time_sec": None},   # replace None with measured full-configuration time
]

# Example after editing:
# speedup_measurements = [
#     {"executors": 1, "time_sec": 120.5},
#     {"executors": 7, "time_sec": 31.2},
# ]

if all(row["time_sec"] is not None for row in speedup_measurements):
    baseline_time = speedup_measurements[0]["time_sec"]
    calculated = []
    for row in speedup_measurements:
        speedup = baseline_time / row["time_sec"]
        efficiency = speedup / row["executors"]
        calculated.append({
            "executors": row["executors"],
            "time_sec": row["time_sec"],
            "speedup": speedup,
            "efficiency": efficiency
        })

    speedup_df = spark.createDataFrame(calculated)
    speedup_df.show(truncate=False)

    full_run = calculated[-1]
    n = full_run["executors"]
    S = full_run["speedup"]

    # Amdahl estimate: S = 1 / ((1 - p) + p/n)
    # Solve for p.
    if n > 1 and S > 1:
        parallel_fraction = (1 - 1/S) / (1 - 1/n)
        print("Estimated parallelizable fraction:", parallel_fraction)
else:
    print("Edit speedup_measurements after timing 1-executor and full-executor runs.")

## 10. README-Ready Summary Template

Replace bracketed values with the actual outputs from this notebook.

### Model 2: PCA + Logistic Regression

For the second model, we used PCA for dimensionality reduction followed by Logistic Regression. PCA reduced the original feature vector into a smaller set of principal components before classification. We evaluated multiple PCA dimensions and compared training error against test error to determine whether the model was underfitting or overfitting.

The best PCA model used **[k] principal components**, which explained approximately **[variance]%** of the feature variance. This model achieved **[train accuracy] training accuracy** and **[test accuracy] test accuracy**, with a weighted F1 score of **[test F1]** on the test set.

The fitting graph shows that the model **[underfits / is well-fit / overfits]**. With too few principal components, the model discarded too much information and had higher error. As the number of components increased, performance improved. The best model was selected based on test F1 and test error rather than training accuracy alone.

### Acoustic-State Analysis

We also performed an unsupervised acoustic-state analysis using PCA followed by KMeans clustering. This analysis treats hive sounds as possible hidden sound states rather than only labeled examples. The best KMeans model used **[k] clusters** and achieved a silhouette score of **[score]**. We then compared each acoustic state to the known target labels to see whether hive conditions were associated with recurring sound patterns.

This analysis suggests that **[main cluster insight]**. Some clusters were associated with **[target/class]**, while other clusters contained mixed labels. This means that beehive sounds contain useful structure, but the current features may not fully separate every hive condition. Future work could improve this by extracting more detailed temporal audio features, using longer sound windows, and training sequence-based models.

## 11. Stop Spark Session

In [ ]:
# Uncomment when finished.
# spark.stop()